# Lognormal kappa-spread sweep on the Colab T4

Sweeps heterogeneous capital productivity `kappa ~ LogNormal(mu, sigma)`,
`mu = -sigma^2/2` (mean fixed to 1), over `sigma` from 0 (homogeneous) to 1
(wide spread) -- Aldo Glielmo's guidance (2026-07-25): matching an empirical
wealth Gini is a poorly-identified target not worth chasing, so this sweep
doesn't fit anything. It trains one full KS run per sigma, simulates a long
steady-state rollout, and saves a full figure set per cell plus a
cross-sigma comparison, so the nicest-looking sigma(s) can be picked for the
paper by eye. Full design writeup: `runs/ks-wealth-calibration/README.md`.

Agent kappas are placed on a deterministic, regular quantile mesh (not drawn
i.i.d.) so every cell is exactly reproducible and free of sampling noise --
see the README's "Sampling" section.

Default config: `n_agents=500`, `num_envs=8`, `sigmas=[0.0,0.2,0.4,0.6,0.8,1.0]`
(6 cells, no search -- every cell is trained and kept). **Time the first
cell before assuming the rest fit in one Colab session.** results.csv and
every cell's figures/raw rollout are checkpointed as each cell finishes, so a
disconnect partway through only loses the run in progress.


In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps


In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L


In [ ]:
# Mount Drive BEFORE the run so results are saved as soon as they finish
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-wealth-calibration'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the sweep finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")


## Sigma sweep

Runs `runs/ks-wealth-calibration/config.yaml` as-is: `n_agents=500`,
`device=gpu`, `sigmas=[0.0,0.2,0.4,0.6,0.8,1.0]`. Pass dotlist overrides
after the script path to change any of these -- e.g. a quick CPU smoke test
with two sigmas, or a finer/wider grid:
`!python runs/ks-wealth-calibration/sweep_lognormal.py n_agents=50 device=cpu sim_steps=200 total_timesteps=2000 "sigmas=[0.0,0.5]"`
`!python runs/ks-wealth-calibration/sweep_lognormal.py "sigmas=[0.0,0.15,0.3,0.45,0.6,0.75,0.9,1.0]"`


In [ ]:
!python runs/ks-wealth-calibration/sweep_lognormal.py


In [ ]:
save_results('ks-wealth-calibration')


## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-wealth-calibration/results/results.csv')
display(df[['sigma', 'kappa_std', 'capital_gini', 'top_0.1_share', 'top_0.01_share', 'K_mean', 'euler_mean_abs']])

display(Image('runs/ks-wealth-calibration/results/comparison.png'))


## Per-sigma steady-state dashboards

One dashboard per sigma: kappa profile, aggregate capital/consumption paths,
the aggregate KS shock, the Lorenz curve, and the wealth histogram.


In [ ]:
for sigma in df['sigma']:
    print(f"sigma = {sigma:.2f}")
    display(Image(f'runs/ks-wealth-calibration/results/figures/sigma_{sigma:.2f}_steady_state.png'))
